<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
print(dataset)
print(len(dataset["train"]))

In [ ]:
import time

start = time.time()

for i, row in enumerate(dataset["train"]):
    if i % 100000 == 0:
        print(i, "rows processed")

    if i == 500000:
        break

print("Time:", time.time() - start)

In [ ]:
import unicodedata
import re

MAX_DOCS = 500000

with open("bn_news.txt", "w", encoding="utf-8") as f:

    for i, row in enumerate(dataset["train"]):

        text = unicodedata.normalize("NFC", row["text"])
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            f.write(text + "\n")

        if i + 1 >= MAX_DOCS:
            break

print("Saved", MAX_DOCS, "documents")

In [ ]:
word_count = 0
char_count = 0

with open("bn_news.txt", "r", encoding="utf-8") as f:
    for line in f:
        word_count += len(line.split())
        char_count += len(line)

print("Words:", word_count)
print("Characters:", char_count)
print("Approx MB:", char_count / 1024 / 1024)

In [ ]:
import gc

try:
    del dataset
except:
    pass

gc.collect()

In [ ]:
!pip -q install aksharamukha

In [ ]:
!ls -lh

In [ ]:
count = 0

with open("bn_news.txt", "r", encoding="utf-8") as f:
    for _ in f:
        count += 1

print("Documents:", count)

In [ ]:
import random

random.seed(42)

train_count = 0
test_count = 0

with open("bn_news.txt", "r", encoding="utf-8") as inp, \
     open("train_bn.txt", "w", encoding="utf-8") as train_f, \
     open("test_bn.txt", "w", encoding="utf-8") as test_f:

    for line in inp:

        line = line.strip()

        if not line:
            continue

        if random.random() < 0.1:
            test_f.write(line + "\n")
            test_count += 1
        else:
            train_f.write(line + "\n")
            train_count += 1

print("Train:", train_count)
print("Test:", test_count)

In [ ]:
import os

print("Train MB:", round(os.path.getsize("train_bn.txt")/1024/1024, 2))
print("Test MB :", round(os.path.getsize("test_bn.txt")/1024/1024, 2))

Transliteration

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()

count = 0

with open("train_bn.txt", "r", encoding="utf-8") as fin:
    for line in fin:
        line = line.strip()

        if not line:
            continue

        _ = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        count += 1

        if count == 1000:
            break

elapsed = time.time() - start

print("1000 docs:", elapsed, "seconds")
print("Docs/sec:", count / elapsed)

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()
count = 0

with open("train_bn.txt", "r", encoding="utf-8") as fin, \
     open("train_iso.txt", "w", encoding="utf-8") as fout:

    for line in fin:

        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "HK",
            line
        )

        fout.write(iso + "\n")

        count += 1

        if count % 10000 == 0:
            elapsed = time.time() - start
            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

In [ ]:
from aksharamukha import transliterate
import time

start = time.time()
count = 0

with open("test_bn.txt", "r", encoding="utf-8") as fin, \
     open("test_iso.txt", "w", encoding="utf-8") as fout:

    for line in fin:

        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "HK",
            line
        )

        fout.write(iso + "\n")

        count += 1

        if count % 10000 == 0:
            elapsed = time.time() - start
            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

In [ ]:
for fname in [
    "train_bn.txt",
    "train_iso.txt"
]:
    print("\n---", fname, "---")

    with open(fname, "r", encoding="utf-8") as f:
        for _ in range(3):
            print(next(f).strip())

In [ ]:
import os

print("train_bn.txt :", round(os.path.getsize("train_bn.txt")/1024/1024,2), "MB")
print("train_iso.txt:", round(os.path.getsize("train_iso.txt")/1024/1024,2), "MB")

print("test_bn.txt :", round(os.path.getsize("test_bn.txt")/1024/1024,2), "MB")
print("test_iso.txt:", round(os.path.getsize("test_iso.txt")/1024/1024,2), "MB")

In [ ]:
from collections import Counter

def corpus_stats(filepath):
    total_chars = 0
    total_words = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            words = line.split()

            total_chars += len(line)
            total_words += len(words)

    avg_word_len = total_chars / total_words

    return {
        "chars": total_chars,
        "words": total_words,
        "avg_word_len": avg_word_len
    }

bn_stats = corpus_stats("train_bn.txt")
ascii_stats = corpus_stats("train_iso.txt")

print("Bengali:", bn_stats)
print("ASCII:", ascii_stats)

In [ ]:
expansion_factor = (
    ascii_stats["chars"] /
    bn_stats["chars"]
)

print("Expansion Factor:", expansion_factor)

In [ ]:
def unique_vocab(filepath):

    vocab = set()

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:

            words = line.split()

            for w in words:
                vocab.add(w)

    return len(vocab)

bn_vocab = unique_vocab("train_bn.txt")
iso_vocab = unique_vocab("train_iso.txt")

print("BN Vocabulary:", bn_vocab)
print("ASCII Vocabulary:", iso_vocab)

In [ ]:
vocab_ratio = iso_vocab / bn_vocab

print("Vocabulary Ratio:", vocab_ratio)

In [ ]:
import re

bengali_pattern = re.compile(r'[\u0980-\u09FF]')

total_lines = 0
converted_lines = 0

with open("train_iso.txt", "r", encoding="utf-8") as f:

    for line in f:

        total_lines += 1

        if not bengali_pattern.search(line):
            converted_lines += 1

coverage = converted_lines / total_lines

print("Coverage:", coverage)

In [ ]:
char_ratio = (
    ascii_stats["chars"] /
    bn_stats["chars"]
)

print("Character Ratio:", char_ratio)

In [ ]:
print("\n=== TRANSLITERATION ANALYSIS ===\n")

print("BN Characters:", bn_stats["chars"])
print("ISO Characters:", ascii_stats["chars"])

print()

print("BN Words:", bn_stats["words"])
print("ISO Words:", ascii_stats["words"])

print()

print("BN Avg Word Length:",
      round(bn_stats["avg_word_len"], 3))

print("ISO Avg Word Length:",
      round(ascii_stats["avg_word_len"], 3))

print()

print("BN Vocabulary:",
      bn_vocab)

print("ISO Vocabulary:",
      iso_vocab)

print()

print("Vocabulary Ratio:",
      round(vocab_ratio, 4))

print("Expansion Factor:",
      round(expansion_factor, 4))

print("Coverage:",
      round(coverage, 4))

In [ ]:
import re

bengali_pattern = re.compile(r'[\u0980-\u09FF]')

count = 0

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:

        if bengali_pattern.search(line):

            print(line[:300])

            count += 1

            if count == 20:
                break

In [ ]:
import re

# Bengali letters only
letter_pattern = re.compile(r'[অ-হািীুূৃেৈোৌংঃঁ]')

total_lines = 0
clean_lines = 0

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:

        total_lines += 1

        if not letter_pattern.search(line):
            clean_lines += 1

coverage = clean_lines / total_lines

print("Letter Coverage:", coverage)

In [ ]:
import re
from collections import Counter

pattern = re.compile(r'[\u0980-\u09FF]')

counter = Counter()

with open("train_iso.txt", encoding="utf-8") as f:

    for line in f:
        counter.update(pattern.findall(line))

print(counter.most_common(20))

In [ ]:
bn_ttr = bn_vocab / bn_stats["words"]
iso_ttr = iso_vocab / ascii_stats["words"]

print("BN TTR :", round(bn_ttr, 6))
print("ISO TTR:", round(iso_ttr, 6))

Tokenization

In [ ]:
!pip -q install tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(
    BPE(
        unk_token="[UNK]"
    )
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=1,
    special_tokens=["[UNK]"]
)

tokenizer.train(
    ["train_bn.txt"],
    trainer
)

tokenizer.save("bpe_bn_5000.json")

print("Vocabulary Size:",
      tokenizer.get_vocab_size())

In [ ]:
sample = "নয়াদিল্লি: আজ ভ্যালেন্টাইনস ডে।"

enc = tokenizer.encode(sample)

print(enc.tokens)

In [ ]:
import numpy as np

def evaluate_tokenizer(tokenizer, filepath):

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            words = line.split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                enc = tokenizer.encode(word)

                n_subwords = len(enc.tokens)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += enc.tokens.count("[UNK]")

            doc_token_counts.append(doc_tokens)

    fertility = total_subwords / total_words

    cpt = total_chars / total_subwords

    compression = total_chars / total_subwords

    oov = unk_tokens / total_subwords

    avg_tokens = np.mean(doc_token_counts)

    wfr = fragmented_words / total_words

    variance = np.var(doc_token_counts)

    return {
        "vocab": tokenizer.get_vocab_size(),
        "oov": oov,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
print(tokenizer.token_to_id("[UNK]"))

In [ ]:
results = evaluate_tokenizer(
    tokenizer,
    "test_bn.txt"
)

print(results)

In [ ]:
from tokenizers.pre_tokenizers import ByteLevel

In [ ]:
samples = [
    "নয়াদিল্লি",
    "বাংলাদেশ",
    "কলকাতা",
    "ভ্যালেন্টাইনস",
    "ক্রিকেটার"
]

for s in samples:
    print(s)
    print(tokenizer.encode(s).tokens)
    print()

In [ ]:
word = "য়"

print(tokenizer.encode(word).tokens)

In [ ]:
print(tokenizer.token_to_id("য়"))

In [ ]:
from tokenizers.trainers import BpeTrainer

bengali_chars = list(
    "অআইঈউঊঋএঐওঔ"
    "কখগঘঙচছজঝঞ"
    "টঠডঢণতথদধন"
    "পফবভমযরলশষসহ"
    "ড়ঢ়য়ৎ"
    "ািীুূৃেৈোৌ"
    "ংঃঁ"
    "০১২৩৪৫৬৭৮৯"
    "।"
)

In [ ]:
tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)

tokenizer.pre_tokenizer = Whitespace()

tokenizer.train(
    ["train_bn.txt"],
    trainer
)

In [ ]:
trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=1,
    special_tokens=["[UNK]"],
    initial_alphabet=bengali_chars
)

In [ ]:
for ch in ["য়", "ড়", "ঢ়", "ৎ"]:
    print(ch, tokenizer.encode(ch).tokens)

In [ ]:
print(tokenizer.get_vocab_size())

In [ ]:
print(tokenizer.token_to_id("য়"))

In [ ]:
print(tokenizer.encode("নয়াদিল্লি").tokens)

In [ ]:
results = evaluate_tokenizer(
    tokenizer,
    "test_bn.txt"
)

print(results)

In [ ]:
import gc
import csv
import numpy as np

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
chars = set()

with open("train_bn.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

bengali_chars = sorted(list(chars))

print("Character inventory:", len(bengali_chars))

In [ ]:
def evaluate_tokenizer(tokenizer, filepath):

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            words = line.split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                enc = tokenizer.encode(word)

                n_subwords = len(enc.tokens)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += enc.tokens.count("[UNK]")

            doc_token_counts.append(doc_tokens)

    fertility = total_subwords / total_words

    cpt = total_chars / total_subwords

    compression = total_chars / total_subwords

    oov = unk_tokens / total_subwords

    avg_tokens = np.mean(doc_token_counts)

    wfr = fragmented_words / total_words

    variance = np.var(doc_token_counts)

    return {
        "vocab": tokenizer.get_vocab_size(),
        "oov": oov,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "bpe_bn_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=bengali_chars
    )

    tokenizer.train(
        ["train_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_bn.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "BPE",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("bpe_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
chars = set()

with open("train_iso.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

iso_chars = sorted(list(chars))

print("Character inventory:", len(iso_chars))
print(iso_chars[:50])

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "bpe_iso_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining ISO BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=iso_chars
    )

    tokenizer.train(
        ["train_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_iso.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "ISO",
            "BPE",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("bpe_iso_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import csv

bn = []
iso = []

with open("bpe_bn_results.csv") as f:
    bn = list(csv.DictReader(f))

with open("bpe_iso_results.csv") as f:
    iso = list(csv.DictReader(f))

print(
    f"{'Vocab':<8}"
    f"{'BN_Fert':<12}"
    f"{'ISO_Fert':<12}"
    f"{'BN_CPT':<12}"
    f"{'ISO_CPT':<12}"
)

for b, i in zip(bn, iso):

    print(
        f"{b['vocab']:<8}"
        f"{float(b['fertility']):<12.4f}"
        f"{float(i['fertility']):<12.4f}"
        f"{float(b['cpt']):<12.4f}"
        f"{float(i['cpt']):<12.4f}"
    )

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "wp_bn_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining Bengali WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=bengali_chars
    )

    tokenizer.train(
        ["train_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_bn_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_bn.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "WordPiece",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
with open("wp_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import gc
import csv

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

results_file = "wp_iso_results.csv"

with open(results_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    print(f"\nTraining ISO WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"],
        initial_alphabet=iso_chars
    )

    tokenizer.train(
        ["train_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_iso_{vocab_size}.json"
    )

    metrics = evaluate_tokenizer(
        tokenizer,
        "test_iso.txt"
    )

    print(metrics)

    with open(results_file, "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "ISO",
            "WordPiece",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

    del tokenizer
    gc.collect()

print("\nFinished.")

In [ ]:
!pip -q install sentencepiece

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_bn.txt",
        model_prefix=f"uni_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(model_path, filepath):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    total_words = 0
    total_subwords = 0
    total_chars = 0

    fragmented_words = 0
    unk_tokens = 0

    doc_token_counts = []

    with open(filepath, encoding="utf-8") as f:

        for line in f:

            words = line.strip().split()

            doc_tokens = 0

            for word in words:

                total_words += 1
                total_chars += len(word)

                pieces = sp.encode(word, out_type=str)

                n_subwords = len(pieces)

                total_subwords += n_subwords
                doc_tokens += n_subwords

                if n_subwords > 1:
                    fragmented_words += 1

                unk_tokens += pieces.count("<unk>")

            doc_token_counts.append(doc_tokens)

    return {
        "vocab": sp.get_piece_size(),
        "oov": unk_tokens / total_subwords,
        "fertility": total_subwords / total_words,
        "cpt": total_chars / total_subwords,
        "compression": total_chars / total_subwords,
        "avg_tokens": np.mean(doc_token_counts),
        "wfr": fragmented_words / total_words,
        "variance": np.var(doc_token_counts)
    }

In [ ]:
import csv

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

with open("uni_bn_results.csv", "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    metrics = evaluate_unigram(
        f"uni_bn_{vocab_size}.model",
        "test_bn.txt"
    )

    print(metrics)

    with open("uni_bn_results.csv", "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "Unigram",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

print("Finished")

In [ ]:
with open("uni_bn_results.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_iso.txt",
        model_prefix=f"uni_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")

In [ ]:
import csv

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

with open("uni_iso_results.csv", "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

for vocab_size in VOCABS:

    metrics = evaluate_unigram(
        f"uni_iso_{vocab_size}.model",
        "test_iso.txt"
    )

    print(metrics)

    with open("uni_iso_results.csv", "a", newline="") as f:

        writer = csv.writer(f)

        writer.writerow([
            "Bengali",
            "Unigram",
            metrics["vocab"],
            metrics["oov"],
            metrics["fertility"],
            metrics["cpt"],
            metrics["compression"],
            metrics["avg_tokens"],
            metrics["wfr"],
            metrics["variance"]
        ])

print("Finished")


# Table 1. BPE Results (Bangla)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000067 |     1.8266 |     3.0616 |     82.1765 |     0.4867 |     30236.16 |
| 10,000 | 0.000079 |     1.5473 |     3.6142 |     69.6116 |     0.3600 |     21921.07 |
| 15,000 | 0.000085 |     1.4422 |     3.8775 |     64.8853 |     0.3050 |     19086.05 |
| 20,000 | 0.000088 |     1.3864 |     4.0337 |     62.3726 |     0.2749 |     17591.76 |
| 25,000 | 0.000091 |     1.3510 |     4.1393 |     60.7814 |     0.2549 |     16656.31 |
| 30,000 | 0.000092 |     1.3261 |     4.2170 |     59.6613 |     0.2407 |     16000.68 |
| 35,000 | 0.000094 |     1.3080 |     4.2754 |     58.8456 |     0.2301 |     15537.40 |
| 40,000 | 0.000095 |     1.2943 |     4.3207 |     58.2294 |     0.2219 |     15201.74 |
| 45,000 | 0.000095 |     1.2833 |     4.3577 |     57.7348 |     0.2154 |     14914.70 |
| 50,000 | 0.000096 | **1.2745** | **4.3877** | **57.3395** | **0.2102** | **14680.40** |

---

# Table 2. BPE Results (ASCII)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000068 |     1.8132 |     3.6132 |     81.5747 |     0.4838 |     29682.35 |
| 10,000 | 0.000079 |     1.5465 |     4.2364 |     69.5755 |     0.3604 |     21873.62 |
| 15,000 | 0.000085 |     1.4458 |     4.5315 |     65.0437 |     0.3072 |     19136.02 |
| 20,000 | 0.000088 |     1.3924 |     4.7052 |     62.6431 |     0.2777 |     17705.69 |
| 25,000 | 0.000090 |     1.3582 |     4.8236 |     61.1057 |     0.2580 |     16812.32 |
| 30,000 | 0.000092 |     1.3346 |     4.9089 |     60.0428 |     0.2443 |     16185.69 |
| 35,000 | 0.000093 |     1.3177 |     4.9719 |     59.2826 |     0.2342 |     15753.47 |
| 40,000 | 0.000094 |     1.3045 |     5.0224 |     58.6870 |     0.2262 |     15409.39 |
| 45,000 | 0.000095 |     1.2942 |     5.0623 |     58.2235 |     0.2200 |     15144.32 |
| 50,000 | 0.000095 | **1.2858** | **5.0952** | **57.8477** | **0.2149** | **14922.79** |

---

# Table 3. Best BPE Comparison (50k Vocabulary)

| Metric      |       Bangla |        ASCII | Better             |
| ----------- | -----------: | -----------: | :----------------- |
| OOV         |     0.000096 | **0.000095** | ASCII (negligible) |
| Fertility   |   **1.2745** |       1.2858 | **Bangla**         |
| CPT         |       4.3877 |   **5.0952** | ASCII*             |
| Avg. Tokens |  **57.3395** |      57.8477 | **Bangla**         |
| WFR         |   **0.2102** |       0.2149 | **Bangla**         |
| Variance    | **14680.40** |     14922.79 | **Bangla**         |

Higher CPT is expected for ASCII because transliterated words contain more Latin characters per token and does not necessarily indicate superior segmentation.



## Table 4. WordPiece Results (Bangla)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000035 |     2.0689 |     2.7030 |     93.0775 |     0.5676 |     38447.32 |
| 10,000 | 0.000044 |     1.6381 |     3.4138 |     73.6977 |     0.3964 |     24549.05 |
| 15,000 | 0.000049 |     1.4991 |     3.7303 |     67.4442 |     0.3295 |     20647.21 |
| 20,000 | 0.000051 |     1.4276 |     3.9172 |     64.2270 |     0.2932 |     18697.77 |
| 25,000 | 0.000053 |     1.3833 |     4.0426 |     62.2353 |     0.2698 |     17527.70 |
| 30,000 | 0.000054 |     1.3526 |     4.1344 |     60.8529 |     0.2530 |     16734.56 |
| 35,000 | 0.000055 |     1.3302 |     4.2040 |     59.8456 |     0.2407 |     16149.79 |
| 40,000 | 0.000055 |     1.3135 |     4.2575 |     59.0930 |     0.2313 |     15696.74 |
| 45,000 | 0.000056 |     1.3004 |     4.3005 |     58.5032 |     0.2238 |     15373.75 |
| 50,000 | 0.000056 | **1.2896** | **4.3363** | **58.0200** | **0.2177** | **15091.11** |

---

## Table 5. WordPiece Results (ASCII)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000036 |     2.0659 |     3.1714 |     92.9405 |     0.5641 |     38218.98 |
| 10,000 | 0.000045 |     1.6502 |     3.9702 |     74.2404 |     0.3992 |     24832.73 |
| 15,000 | 0.000049 |     1.5122 |     4.3326 |     68.0303 |     0.3322 |     20938.98 |
| 20,000 | 0.000051 |     1.4406 |     4.5479 |     64.8096 |     0.2966 |     18989.98 |
| 25,000 | 0.000053 |     1.3959 |     4.6933 |     62.8013 |     0.2736 |     17813.44 |
| 30,000 | 0.000054 |     1.3656 |     4.7976 |     61.4360 |     0.2571 |     17043.92 |
| 35,000 | 0.000055 |     1.3434 |     4.8769 |     60.4378 |     0.2453 |     16436.90 |
| 40,000 | 0.000056 |     1.3269 |     4.9374 |     59.6968 |     0.2360 |     16000.91 |
| 45,000 | 0.000056 |     1.3136 |     4.9873 |     59.0993 |     0.2286 |     15658.77 |
| 50,000 | 0.000057 | **1.3031** | **5.0277** | **58.6248** | **0.2225** | **15391.12** |

---

## Table 6. Best WordPiece Comparison (50k Vocabulary)

| Metric      |       Bangla |      ASCII | Better                             |
| ----------- | -----------: | ---------: | :--------------------------------- |
| OOV         | **0.000056** |   0.000057 | **Bangla** (difference negligible) |
| Fertility   |   **1.2896** |     1.3031 | **Bangla**                         |
| CPT         |       4.3363 | **5.0277** | ASCII*                             |
| Avg. Tokens |  **58.0200** |    58.6248 | **Bangla**                         |
| WFR         |   **0.2177** |     0.2225 | **Bangla**                         |
| Variance    | **15091.11** |   15391.12 | **Bangla**                         |

Higher CPT for ASCII is expected because transliterated words contain more Latin characters per token; it reflects the script representation rather than better segmentation quality.


## Table 7. SentencePiece Unigram Results (Bangla)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000000 |     1.8507 |     3.0216 |     83.2639 |     0.4542 |     31798.45 |
| 10,000 | 0.000000 |     1.5779 |     3.5440 |     70.9907 |     0.3534 |     23360.10 |
| 15,000 | 0.000000 |     1.4746 |     3.7924 |     66.3400 |     0.3072 |     20361.71 |
| 20,000 | 0.000000 |     1.4180 |     3.9437 |     63.7961 |     0.2812 |     18802.60 |
| 25,000 | 0.000000 |     1.3825 |     4.0450 |     62.1985 |     0.2648 |     17812.11 |
| 30,000 | 0.000000 |     1.3586 |     4.1163 |     61.1207 |     0.2531 |     17157.81 |
| 35,000 | 0.000000 |     1.3408 |     4.1709 |     60.3201 |     0.2451 |     16647.59 |
| 40,000 | 0.000000 |     1.3281 |     4.2106 |     59.7519 |     0.2394 |     16328.78 |
| 45,000 | 0.000000 |     1.3177 |     4.2440 |     59.2815 |     0.2347 |     16033.63 |
| 50,000 | 0.000000 | **1.3097** | **4.2699** | **58.9222** | **0.2310** | **15773.49** |

---

## Table 8. SentencePiece Unigram Results (ASCII)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |     Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | -----------: |
|  5,000 | 0.000000 |     1.9291 |     3.3961 |     86.7901 |     0.4621 |     34133.06 |
| 10,000 | 0.000000 |     1.6522 |     3.9652 |     74.3328 |     0.3641 |     25424.69 |
| 15,000 | 0.000000 |     1.5441 |     4.2429 |     69.4688 |     0.3219 |     22206.34 |
| 20,000 | 0.000000 |     1.4855 |     4.4103 |     66.8321 |     0.2983 |     20491.01 |
| 25,000 | 0.000000 |     1.4488 |     4.5220 |     65.1806 |     0.2838 |     19407.52 |
| 30,000 | 0.000000 |     1.4234 |     4.6028 |     64.0366 |     0.2738 |     18718.90 |
| 35,000 | 0.000000 |     1.4055 |     4.6615 |     63.2302 |     0.2666 |     18185.60 |
| 40,000 | 0.000000 |     1.3923 |     4.7057 |     62.6369 |     0.2614 |     17824.16 |
| 45,000 | 0.000000 |     1.3822 |     4.7399 |     62.1846 |     0.2573 |     17541.88 |
| 50,000 | 0.000000 | **1.3736** | **4.7695** | **61.7988** | **0.2540** | **17306.75** |

---

## Table 9. Best SentencePiece Unigram Comparison (50k Vocabulary)

| Metric      |       Bangla |        ASCII | Better     |
| ----------- | -----------: | -----------: | :--------- |
| OOV         | **0.000000** | **0.000000** | Tie        |
| Fertility   |   **1.3097** |       1.3736 | **Bangla** |
| CPT         |       4.2699 |   **4.7695** | ASCII*     |
| Avg. Tokens |  **58.9222** |      61.7988 | **Bangla** |
| WFR         |   **0.2310** |       0.2540 | **Bangla** |
| Variance    | **15773.49** |     17306.75 | **Bangla** |

Higher CPT for ASCII is expected because transliterated words contain more Latin characters per token and does not indicate superior tokenization quality.

---

## Table 10. Overall Best Tokenizer Comparison (50k Vocabulary)

| Tokenizer                 | Script |          OOV |  Fertility |    CPT | Avg. Tokens |        WFR |     Variance |
| ------------------------- | ------ | -----------: | ---------: | -----: | ----------: | ---------: | -----------: |
| **BPE**                   | Bangla |     0.000096 | **1.2745** | 4.3877 | **57.3395** | **0.2102** | **14680.40** |
| **BPE**                   | ASCII  |     0.000095 |     1.2858 | 5.0952 |     57.8477 |     0.2149 |     14922.79 |
| **WordPiece**             | Bangla |     0.000056 |     1.2896 | 4.3363 |     58.0200 |     0.2177 |     15091.11 |
| **WordPiece**             | ASCII  |     0.000057 |     1.3031 | 5.0277 |     58.6248 |     0.2225 |     15391.12 |
| **SentencePiece Unigram** | Bangla | **0.000000** |     1.3097 | 4.2699 |     58.9222 |     0.2310 |     15773.49 |
| **SentencePiece Unigram** | ASCII  | **0.000000** |     1.3736 | 4.7695 |     61.7988 |     0.2540 |     17306.75 |

### Key observations

* **BPE** achieved the best overall intrinsic performance with the **lowest fertility, WFR, average tokens, and variance**.
* **WordPiece** ranked second and consistently outperformed SentencePiece Unigram.
* **SentencePiece Unigram** achieved **zero OOV** but produced the highest fragmentation and variance.
* Across all three tokenizers, the **native Bangla script consistently outperformed ASCII transliteration** in terms of segmentation compactness and stability.
* **ASCII transliteration** naturally resulted in higher **Characters per Token (CPT)** because transliterated words are represented using longer Latin character sequences. This increase reflects the script representation rather than improved tokenization quality.
